# 02 — Ingesta Bronze con Auto Loader (`cloudFiles`)
Ingesta incremental de los archivos JSON multi-línea desde el Volumen de Unity Catalog hacia `bronze.pdac.raw_triplets` preservando la trazabilidad completa.

**1. Configuración de rutas, checkpoint y tabla destino**
Define las constantes del catálogo `bronze`, el Volumen origen y las subcarpetas dedicadas de control para Auto Loader.

In [ ]:
from pyspark.sql import functions as F

# Catálogo, esquema y Volumen definidos en 01_setup.ipynb
CATALOG = "bronze"
SCHEMA = "pdac"
VOLUME_ROOT = f"/Volumes/{CATALOG}/{SCHEMA}/raw_triplets_vol"

RAW_INPUT_PATH = VOLUME_ROOT
CHECKPOINT_PATH = f"{VOLUME_ROOT}/_checkpoints/bronze_triplets"
SCHEMA_PATH = f"{VOLUME_ROOT}/_schemas/bronze_triplets"
TARGET_TABLE = f"{CATALOG}.{SCHEMA}.raw_triplets"

# Sugerencia de esquema exacta según los JSON generados en data/03_processed/
SCHEMA_HINTS = (
    "doc_id STRING, "
    "categoria STRING, "
    "tripletas ARRAY<STRUCT<"
    "sujeto: STRING, tipo_sujeto: STRING, relacion: STRING, "
    "objeto: STRING, tipo_objeto: STRING, score: DOUBLE>>"
)

# Mapeo de subcarpetas a dominio documental legible para auditoría
SOURCE_DOMAIN_MAP = {
    "01_guias": "clinical_guidelines",
    "02_papers": "pubmed_papers",
    "03_ensayos": "clinical_trials",
    "04_farmacos": "aemps_cima_drugs",
}

print(f"Origen Volumen UC : {RAW_INPUT_PATH}")
print(f"Checkpoint path   : {CHECKPOINT_PATH}")
print(f"Schema location   : {SCHEMA_PATH}")
print(f"Tabla Delta       : {TARGET_TABLE}")

**2. Inspección previa del Volumen en Unity Catalog**
Verifica las subcarpetas y el número de archivos JSON disponibles en el Volumen antes de lanzar la ingesta.

In [ ]:
def inspect_uc_volume(path: str) -> None:
    """Lista subcarpetas de datos y cuenta archivos JSON ignorando directorios internos (_*)."""
    entries = dbutils.fs.ls(path)
    data_dirs = [e for e in entries if e.isDir() and not e.name.startswith("_")]
    root_jsons = [e for e in entries if not e.isDir() and e.name.endswith(".json")]
    total_files = len(root_jsons)
    print(f"Archivos JSON en raíz: {len(root_jsons)}")
    for folder in sorted(data_dirs, key=lambda x: x.name):
        sub_entries = dbutils.fs.ls(folder.path)
        json_files = [s for s in sub_entries if not s.isDir() and s.name.endswith(".json")]
        nested_dirs = [s for s in sub_entries if s.isDir() and not s.name.startswith("_")]
        for nd in nested_dirs:
            json_files.extend(
                s for s in dbutils.fs.ls(nd.path) if not s.isDir() and s.name.endswith(".json")
            )
        total_files += len(json_files)
        print(f"  - {folder.name:<18} -> {len(json_files)} archivos JSON")
    print(f"Total JSON listos para Auto Loader: {total_files}")


inspect_uc_volume(RAW_INPUT_PATH)

**3. Lectura incremental con Auto Loader (`cloudFiles`)**
Configura el stream JSON en modo `multiLine=true` con inferencia asistida y columna de rescate para evolución de esquema.

In [ ]:
def read_raw_triplets_stream(input_path: str, schema_location: str, schema_hints: str):
    """Crea un DataFrame streaming con Auto Loader compatible con JSON multi-línea en Unity Catalog."""
    stream_df = (
        spark.readStream.format("cloudFiles")
        .option("cloudFiles.format", "json")
        .option("multiLine", "true")
        .option("pathGlobFilter", "*.json")
        .option("cloudFiles.schemaLocation", schema_location)
        .option("cloudFiles.inferColumnTypes", "true")
        .option("cloudFiles.schemaHints", schema_hints)
        .option("cloudFiles.schemaEvolutionMode", "rescue")
        .option("rescuedDataColumn", "_rescued_data")
        .load(input_path)
    )
    # Filtro defensivo para excluir carpetas de control internas si existieran
    return stream_df.filter(
        ~F.col("_metadata.file_path").contains("/_checkpoints/")
        & ~F.col("_metadata.file_path").contains("/_schemas/")
    )


raw_stream_df = read_raw_triplets_stream(RAW_INPUT_PATH, SCHEMA_PATH, SCHEMA_HINTS)

**4. Despliegue de tripletas y enriquecimiento de auditoría**
Explota el array `tripletas` sin descartar ningún registro y añade metadatos de archivo, fuente documental y marca temporal.

In [ ]:
def enrich_bronze_triplets(stream_df, domain_map: dict):
    """Aplana las tripletas preservando registros vacíos/malformados y agrega trazabilidad."""
    domain_expr = F.create_map([F.lit(item) for kv in domain_map.items() for item in kv])
    extracted_folder = F.regexp_extract(
        F.col("_metadata.file_path"), r"/(0[1-4]_[a-z]+)/", 1
    )
    source_folder_col = F.coalesce(
        F.when(F.length(extracted_folder) > 0, extracted_folder),
        F.col("categoria"),
        F.lit("unknown"),
    )

    exploded_df = stream_df.select(
        F.col("doc_id").cast("string").alias("doc_id"),
        F.col("categoria").cast("string").alias("categoria"),
        F.explode_outer("tripletas").alias("t"),
        F.col("_metadata.file_path").alias("_source_file"),
        F.col("_metadata.file_name").alias("_file_name"),
        F.col("_metadata.file_size").alias("_file_size"),
        F.col("_metadata.file_modification_time").alias("_file_modification_time"),
        source_folder_col.alias("_source_folder"),
        F.col("_rescued_data").alias("_rescued_data"),
    )

    return exploded_df.select(
        F.col("doc_id"),
        F.col("categoria"),
        F.col("t.sujeto").alias("sujeto"),
        F.col("t.tipo_sujeto").alias("tipo_sujeto"),
        F.col("t.relacion").alias("relacion"),
        F.col("t.objeto").alias("objeto"),
        F.col("t.tipo_objeto").alias("tipo_objeto"),
        F.col("t.score").cast("double").alias("score"),
        F.col("_source_file"),
        F.col("_file_name"),
        F.col("_file_size"),
        F.col("_file_modification_time"),
        F.col("_source_folder"),
        F.coalesce(F.element_at(domain_expr, F.col("_source_folder")), F.lit("unknown")).alias("_source_domain"),
        F.current_timestamp().alias("_ingestion_timestamp"),
        F.col("_rescued_data"),
    )


bronze_stream_df = enrich_bronze_triplets(raw_stream_df, SOURCE_DOMAIN_MAP)

**5. Persistencia incremental en Delta Lake (`trigger(availableNow=True)`)**
Ejecuta el stream en modo batch incremental, espera su finalización y registra el comentario de la tabla en Unity Catalog.

In [ ]:
query = (
    bronze_stream_df.writeStream.format("delta")
    .outputMode("append")
    .option("checkpointLocation", CHECKPOINT_PATH)
    .option("mergeSchema", "true")
    .trigger(availableNow=True)
    .toTable(TARGET_TABLE)
)

query.awaitTermination()

progress = query.lastProgress
if progress:
    print(f"Batch ID         : {progress.get('batchId')}")
    print(f"Filas procesadas : {progress.get('numInputRows')}")
else:
    print("Stream finalizado sin nuevos archivos pendientes.")

spark.sql(
    f"""COMMENT ON TABLE {TARGET_TABLE} IS
    'Capa Bronze: Tripletas biomédicas crudas ingeridas incrementalmente mediante Auto Loader con trazabilidad de origen.'"""
)

**6. Verificación post-ingesta y perfilado inicial**
Valida el conteo total de registros, archivos procesados, filas rescatadas y distribución por origen y relación.

In [ ]:
bronze_df = spark.table(TARGET_TABLE)

total_rows = bronze_df.count()
total_files = bronze_df.select("_source_file").distinct().count()
rescued_rows = bronze_df.filter(F.col("_rescued_data").isNotNull()).count()

print(f"Total registros en {TARGET_TABLE} : {total_rows}")
print(f"Archivos fuente únicos ingeridos     : {total_files}")
print(f"Registros con _rescued_data          : {rescued_rows}")

print("\n--- Distribución por carpeta y dominio fuente ---")
bronze_df.groupBy("_source_folder", "_source_domain").agg(
    F.count("*").alias("total_tripletas"),
    F.countDistinct("_source_file").alias("archivos"),
).orderBy("_source_folder").show(truncate=False)

print("--- Distribución cruda por relación ---")
bronze_df.groupBy("relacion").count().orderBy(F.desc("count")).show(truncate=False)

display(bronze_df.limit(10))